# Shoplifting Detection: Pretrained VideoMAE vs. From-Scratch 3D CNN
**Pipeline:** download data → label & split → frame sampling + augmentation → fine-tune pretrained VideoMAE (2-class head) → train scratch baseline → compare F1 / size / speed.

> Run on a GPU runtime (Colab: Runtime → Change runtime type → T4 or better).

In [1]:
!pip -q install transformers accelerate gdown opencv-python-headless scikit-learn pandas matplotlib seaborn

In [2]:
import os, glob, random, time, copy, zipfile, subprocess
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt, seaborn as sns

SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 1. Download and inspect the dataset

In [3]:
FILE_ID = "1KCKfyIGbQi8a7bIYta3LM8dFStxVzVX-"
if not os.path.exists("Shop DataSet.zip"):
    !gdown --id {FILE_ID} -O "Shop DataSet.zip"
if not os.path.exists("data"):
    with zipfile.ZipFile("Shop DataSet.zip") as z: z.extractall("data")

# Show folder structure (first levels) so you can verify labels
for root, dirs, files in os.walk("data"):
    depth = root.count(os.sep)
    if depth <= 3:
        print("  "*depth + os.path.basename(root), f"({len(files)} files)")

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1KCKfyIGbQi8a7bIYta3LM8dFStxVzVX-
From (redirected): https://drive.google.com/uc?id=1KCKfyIGbQi8a7bIYta3LM8dFStxVzVX-&confirm=t&uuid=0541f5ed-dc5c-4981-abe3-8be10b37a528
To: /kaggle/working/Shop DataSet.zip
100%|███████████████████████████████████████| 1.78G/1.78G [00:13<00:00, 128MB/s]
data (0 files)
  Shop DataSet (0 files)
    shop lifters (324 files)
    non shop lifters (531 files)


## 2. Build the labeled table
Labels are inferred from the **folder name** of each video. Edit `is_theft()` if your folders are named differently.

In [5]:
VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".mpg", ".mpeg")

def is_theft(path):
    # label comes from the immediate parent folder: "shop lifters" -> 1, "non shop lifters" -> 0
    return int(os.path.basename(os.path.dirname(path)).strip().lower() == "shop lifters")

paths = [p for p in glob.glob("data/**/*", recursive=True) if p.lower().endswith(VIDEO_EXT)]
df = pd.DataFrame({"path": paths, "label": [is_theft(p) for p in paths]})
print(df.label.value_counts().rename({0: "normal", 1: "shoplifting"}))   # expect 531 normal / 324 shoplifting
assert df.label.nunique() == 2, "Only one class found - fix is_theft() to match your folder names"

train_df, tmp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp.label, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

label
normal         531
shoplifting    324
Name: count, dtype: int64
598 128 129


## 3. Frame sampling + augmentation
**Reuse your Week 5 pipeline here.** The version below is the standard one: segment-based uniform sampling (random offset inside each segment for training, segment centre for eval), then clip-consistent augmentation (same crop/flip/colour on every frame of a clip).

In [6]:
NUM_FRAMES, IMG_SIZE = 16, 224        # VideoMAE-base expects 16 x 224 x 224
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]   # same stats VideoMAE's processor uses

def sample_indices(n_total, n_frames, train):
    seg = n_total / n_frames
    if train:  return [min(int(seg*i + random.random()*seg), n_total-1) for i in range(n_frames)]
    return [min(int(seg*i + seg/2), n_total-1) for i in range(n_frames)]

def read_frames(path, n_frames, train):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if total > 0:
        for i in sample_indices(total, n_frames, train):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ok, f = cap.read()
            if not ok: f = frames[-1] if frames else np.zeros((IMG_SIZE,IMG_SIZE,3), np.uint8)
            else: f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
            frames.append(cv2.resize(f, (256,256)) if f.shape[:2] != (256,256) else f)
    cap.release()
    while len(frames) < n_frames:   # corrupt / very short video fallback
        frames.append(frames[-1] if frames else np.zeros((256,256,3), np.uint8))
    return np.stack(frames)          # T,H,W,C uint8



train_tf = v2.Compose([v2.RandomResizedCrop(IMG_SIZE, scale=(0.7,1.0), antialias=True),
                       v2.RandomHorizontalFlip(),
                       v2.ColorJitter(0.2,0.2,0.2)])
eval_tf  = v2.Compose([v2.Resize(IMG_SIZE, antialias=True), v2.CenterCrop(IMG_SIZE)])
norm = v2.Normalize(MEAN, STD)

class ClipDataset(Dataset):
    def __init__(self, df, train): self.df, self.train = df.reset_index(drop=True), train
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = torch.from_numpy(read_frames(r.path, NUM_FRAMES, self.train)).permute(0,3,1,2).float()/255.  # T,C,H,W
        x = (train_tf if self.train else eval_tf)(x)
        x = norm(x)
        return x, int(r.label)

BS = 4
mk = lambda d, tr: DataLoader(ClipDataset(d, tr), batch_size=BS, shuffle=tr, num_workers=2, pin_memory=True)
train_dl, val_dl, test_dl = mk(train_df, True), mk(val_df, False), mk(test_df, False)

## 4. Load the pretrained model from Hugging Face and adapt the head
`MCG-NJU/videomae-base-finetuned-kinetics`: VideoMAE-B (ViT-B, ~87M params) pretrained with self-supervised masked autoencoding and fine-tuned on Kinetics-400. We replace the 400-class head with a 2-class head.

In [7]:
from transformers import VideoMAEForVideoClassification
CKPT = "MCG-NJU/videomae-base-finetuned-kinetics"
id2label = {0:"normal", 1:"shoplifting"}; label2id = {v:k for k,v in id2label.items()}

vmae = VideoMAEForVideoClassification.from_pretrained(
    CKPT, num_labels=2, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True).to(DEVICE)

# Verify head adaptation
assert vmae.classifier.out_features == 2, "Head is not 2-class!"
print("Classifier head:", vmae.classifier)
with torch.no_grad():
    print("Logits shape:", vmae(pixel_values=torch.randn(1,16,3,224,224,device=DEVICE)).logits.shape)  # (1, 2)

config.json:   0%|          | 0.00/22.9k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `400`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/162 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.query.bias | MISSING    |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.value.bi

Classifier head: Linear(in_features=768, out_features=2, bias=True)
Logits shape: torch.Size([1, 2])


## 5. Shared training / evaluation utilities

In [8]:
class_w = torch.tensor(len(train_df)/(2*np.bincount(train_df.label)), dtype=torch.float32, device=DEVICE)
print("Class weights (normal, shoplifting):", class_w.tolist())

def logits_of(model, x):
    out = model(pixel_values=x) if hasattr(model, "videomae") else model(x)
    return out.logits if hasattr(out, "logits") else out

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); ys, ps = [], []
    for x, y in dl:
        with torch.autocast(DEVICE, enabled=(DEVICE=="cuda")):
            p = logits_of(model, x.to(DEVICE)).argmax(1).cpu()
        ys += y.tolist(); ps += p.tolist()
    return dict(acc=accuracy_score(ys,ps), precision=precision_score(ys,ps,zero_division=0),
                recall=recall_score(ys,ps,zero_division=0), f1=f1_score(ys,ps,zero_division=0)), ys, ps

def fit(model, name, epochs, lr, wd=0.05, params=None):
    opt = torch.optim.AdamW(params or model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs*len(train_dl), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=(DEVICE=="cuda"))
    best, best_state, hist = -1, None, []
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(DEVICE, enabled=(DEVICE=="cuda")):
                loss = F.cross_entropy(logits_of(model, x), y, weight=class_w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        m,_,_ = evaluate(model, val_dl)
        hist.append(dict(epoch=ep+1, train_loss=tot/len(train_dl), **{f"val_{k}":v for k,v in m.items()}))
        print(f"[{name}] ep{ep+1}/{epochs} loss={tot/len(train_dl):.4f} val_F1={m['f1']:.4f} val_acc={m['acc']:.4f}")
        if m["f1"] > best: best, best_state = m["f1"], copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return pd.DataFrame(hist), time.time()-t0

@torch.no_grad()
def benchmark(model, runs=30):
    model.eval(); x = torch.randn(1,NUM_FRAMES,3,IMG_SIZE,IMG_SIZE, device=DEVICE)
    for _ in range(5): logits_of(model, x)
    if DEVICE=="cuda": torch.cuda.synchronize()
    t=time.time()
    for _ in range(runs): logits_of(model, x)
    if DEVICE=="cuda": torch.cuda.synchronize()
    return (time.time()-t)/runs*1000   # ms per clip

def size_info(model):
    n = sum(p.numel() for p in model.parameters())
    return n/1e6, n*4/1e6   # M params, MB (fp32)

Class weights (normal, shoplifting): [0.8059298992156982, 1.3171806335449219]


## 6. Fine-tune pretrained VideoMAE

In [9]:
seed_all()
hist_vm, time_vm = fit(vmae, "VideoMAE", epochs=8, lr=3e-5)

[VideoMAE] ep1/8 loss=0.6661 val_F1=0.5915 val_acc=0.7734
[VideoMAE] ep2/8 loss=0.5767 val_F1=0.8090 val_acc=0.8672
[VideoMAE] ep3/8 loss=0.4047 val_F1=0.8989 val_acc=0.9297
[VideoMAE] ep4/8 loss=0.2825 val_F1=0.8706 val_acc=0.9141
[VideoMAE] ep5/8 loss=0.2518 val_F1=0.8148 val_acc=0.8828
[VideoMAE] ep6/8 loss=0.1502 val_F1=0.9565 val_acc=0.9688
[VideoMAE] ep7/8 loss=0.1122 val_F1=0.9333 val_acc=0.9531
[VideoMAE] ep8/8 loss=0.1142 val_F1=0.9333 val_acc=0.9531
